# OpenAI

**OpenAI** pioneered mainstream LLM APIs with GPT-3 and ChatGPT. Today it remains the most widely used frontier provider for developers. Its **Chat Completions API** is the **de facto reference design** — many competitors, cloud gateways, and local servers (Ollama, vLLM) mimic its request and response shape.

This notebook covers OpenAI's model families, API design, key features, and how this course integrates OpenAI through console scripts and a FastAPI application. For the cross-provider overview, see `01. LLM Providers.ipynb`.


---

## 1. OpenAI in the LLM Ecosystem

OpenAI's influence extends beyond its own products:

- **ChatGPT** brought LLMs to hundreds of millions of users
- The **chat completions** pattern (`messages`, `roles`, `stream=True`) became an industry standard
- **Function / tool calling** shaped how agents invoke external APIs
- **Embeddings APIs** power semantic search and RAG pipelines worldwide

For learning and prototyping, OpenAI is often the default choice because documentation, tutorials, and third-party tools are the most mature. Production teams may still route some traffic to other providers for cost, compliance, or capability reasons.


---

## 2. Model Families

OpenAI maintains several model lines optimized for different tasks and price points. Model IDs and availability change over time — always check the [official models page](https://platform.openai.com/docs/models).

| Model family | Typical use | Notes |
|--------------|-------------|-------|
| **GPT-4o** | General chat, code, vision | Flagship multimodal model; strong all-rounder |
| **GPT-4o-mini** | High-volume, cost-sensitive tasks | Fast and inexpensive; great for learning |
| **o-series (o1, o3, etc.)** | Complex reasoning, math, code | Uses internal reasoning tokens; higher latency and cost |
| **text-embedding-3-small / large** | Semantic search, RAG | Separate embeddings endpoint, not chat |
| **Whisper / TTS** | Speech-to-text, text-to-speech | Audio APIs separate from chat |

### 2.1 Choosing a model

| Task | Suggested starting point |
|------|-------------------------|
| Learning the API | `gpt-4o-mini` |
| Production chatbot | `gpt-4o` or `gpt-4o-mini` depending on quality needs |
| Document Q&A with RAG | `gpt-4o-mini` for retrieval answers; upgrade if quality insufficient |
| Hard reasoning / math | o-series reasoning models |
| Semantic search | `text-embedding-3-small` |

Smaller models reduce cost and latency; frontier models improve quality on nuanced tasks. Measure on your own data rather than assuming you always need the largest model.


---

## 3. API Design

### 3.1 Endpoints and authentication

| Item | Value |
|------|-------|
| **Base URL** | `https://api.openai.com/v1` |
| **Override** | Set `OPENAI_BASE_URL` for proxies, Azure OpenAI, or compatible gateways |
| **Auth** | `Authorization: Bearer <OPENAI_API_KEY>` |
| **Primary endpoint** | `POST /v1/chat/completions` |
| **Python SDK** | `pip install openai` |

### 3.2 Minimal chat completion

```python
from openai import OpenAI

client = OpenAI()  # reads OPENAI_API_KEY from environment

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is an API in one sentence?"},
    ],
    temperature=0.3,
    max_tokens=150,
)

print(response.choices[0].message.content)
print(response.usage)  # prompt_tokens, completion_tokens, total_tokens
```

### 3.3 Message roles

| Role | Purpose |
|------|---------|
| `system` | Instructions and behavior guidelines for the model |
| `user` | End-user input |
| `assistant` | Previous model replies (for multi-turn history) |
| `tool` | Results returned from function/tool calls |

The API is **stateless**: you send the full message list on every request. The model does not remember prior calls unless you include them in `messages`.


---

## 4. Key API Features

### 4.1 Streaming

Set `stream=True` to receive tokens as they are generated. Same final text and billing; better UX for chat interfaces.

```python
stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "List three uses of streaming."}],
    stream=True,
)
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
```

### 4.2 Function and tool calling

The model can return structured JSON describing a function to call (e.g. `get_weather(city="London")`). Your application executes the function and sends the result back. This is the foundation for **LLM agents** and tool use.

### 4.3 JSON mode

`response_format={"type": "json_object"}` encourages valid JSON output — useful for structured extraction pipelines.

### 4.4 Vision (multimodal)

GPT-4o accepts image content in messages via URL or base64. One model can reason over text and images in a single request.

### 4.5 Usage and cost tracking

Every non-streaming response includes a `usage` object with `prompt_tokens`, `completion_tokens`, and `total_tokens`. Use **tiktoken** locally to estimate prompt size before sending.


---

## 5. Inference Parameters

| Parameter | Effect |
|-----------|--------|
| `temperature` | Randomness; 0 = deterministic, higher = more varied |
| `max_tokens` | Cap on completion length |
| `top_p` | Nucleus sampling — alternative to temperature |
| `stop` | Stop sequences that end generation early |
| `presence_penalty` / `frequency_penalty` | Reduce repetition |

For factual Q&A, use low temperature (0–0.3). For creative writing, increase temperature (0.7–1.0). See GenAI Fundamentals — *Inference Parameters and Text Generation* for full detail.


---

## 6. Rate Limits, Errors, and Retries

OpenAI enforces **rate limits** per model and tier. When exceeded, the API returns **HTTP 429**.

| Error | Meaning | Action |
|-------|---------|--------|
| 401 | Invalid API key | Check `OPENAI_API_KEY` in `.env` |
| 429 | Rate limit | Exponential backoff and retry |
| 400 | Bad request | Verify model name and request body |
| 500 | Server error | Retry; check [status page](https://status.openai.com/) |


---

## 7. Azure OpenAI Service

Microsoft hosts OpenAI models on **Azure** for enterprises needing regional data residency, Azure AD, and private networking. The same `openai` Python SDK works with a different `base_url` and deployment-specific model names.


---

## 8. Hands-On in This Course

| Path | Location | What you learn |
|------|----------|----------------|
| **Console** | `../02. OpenAI/01. Console Application/` | Direct SDK — chat, streaming, tiktoken |
| **FastAPI** | `../02. OpenAI/02. Fast API Application/` | Production HTTP API, retries, SSE |

**Setup:** Create an account at [platform.openai.com](https://platform.openai.com), generate an API key, set `OPENAI_API_KEY` in repo root `.env`.

```bash
python "../02. OpenAI/01. Console Application/01_chat_completion.py"
cd "../02. OpenAI/02. Fast API Application"
uvicorn app.main:app --reload --port 8010
```


---

## 9. Summary

OpenAI offers the reference **chat completions API**. Use **GPT-4o-mini** to learn; scale model size based on measured quality. Messages are stateless, streaming improves UX, and production code should handle rate limits with backoff. This course demonstrates OpenAI via console scripts and FastAPI.
